## Ch13. Processing Sequences Using RNNs and CNNs
- RNN은 어느 정도까지 미래를 예측할 수 있는 신경망
- 과거의 패턴이 미래에도 유지된다는 가정 필요
- 활용: 시계열 분석, 임의 길이의 시퀀스 처리 등
- RNN의 약점 두 가지는 불안정한 그래디언트와 짧은 단기 기억
- 트랜스포머가 지금의 최고 수준이며, 이 장은 그 전 단계

#### 순환 뉴런과 층
- RNN은 이전 스텝의 출력을 다시 입력으로 받는 뒤로 향하는 연결이 있는 신경망이다.
- 순환 층의 출력은 ŷ(t) = φ(Wₓᵀx(t) + W_ŷᵀŷ(t−1) + b)이고, 가중치는 현재 입력용(Wₓ)과 이전 출력용(W_ŷ)두 종류
- 미니배치 식은 Ŷ(t) = φ([X(t) Ŷ(t−1)]W + b)
- 시간에 따라 펼치면 같은 층을 스텝마다 그린 모양이 되고, 출력이 과거의 모든 입력에 의존해 기억이 생김.

#### 메모리 셀
- 타임 스텝에 걸쳐 상태를 보존하는 신경망의 일부. 순환 뉴런이나 순환 층이 가장 기본적인 셀.
- 셀 상태는 h(t) = f(x(t), h(t−1)). 출력 ŷ(t)는 보통 현재 상태의 함수.
- 기본 셀은 출력 = 상태이지만, 복잡한 셀은 둘이 다를 수 있다.
- 기본 셀은 약 10스텝 정도의 짧은 패턴만 학습하고, 복잡한 셀은 약 10배 긴 패턴을 학습한다.
- 현대 표준은 은닉 상태를 되먹임하는 Elman식 상태 피드백이다.

#### 입력 시퀀스와 출력 시퀀스
| 구조 | 입력->출력 |
| --- | --- |
| 시퀀스 투 시퀀스 | 시퀀스->시퀀스 |
| 시퀀스 투 벡터 | 시퀀스-> 벡터 하나|
| 벡터 투 시퀀스 | 벡터 하나->시퀀스 |
| 인코더-디코더 | 시퀀스->벡터->시퀀스 |
- 시퀀스 투 벡터는 마지막 출력만 쓰고, 벡터 투 시퀀스는 같은 입력을 매 스텝 반복해 넣는다.
- 인코더-디코더는 문장 전체를 본 뒤에 번역할 수 있어서, 단일 시퀀스 투 시퀀스 RNN보다 번역에 훨씬 좋다.
- 문제는 RNN을 어떻게 훈련하느냐

#### RNN 훈련(BPTT)
- RNN을 시간에 따라 펼친 다음, 일반 역전파를 적용한다. 이 방법을 BPTT(시간에 따른 역전파)라고 한다.
- 흐름은 순전파->손실 계산->역전파->경사 하강 한 스텝
- 손실이 일부 출력을 무시하면 그 출력을 통한 그래디언트 경로는 막힌다.
- 같은 W, b가 모든 스텝에서 공유되므로 그래디언트가 여러 번 합산한다.

#### 시계열 예측
- 시계열의 종류
| 용어 | 뜻 |
| --- | --- |
| 시계열 | 보통 일정한 간격의 서로 다른 시점마다 값이 있는 데이터 |
| 다변량 시계열 | 시점당 값이 여러 개 |
| 단변량 시계열 | 시점당 값이 하나 |
- 시계열의 대표 과제는 예측
- 크기가 다른 시계열을 비교할 때는 MAPE처럼 비율 지표가 낫고, 큰 오차를 크게 벌주려면 MSE
- 차분은 추세와 계절성을 제거해 정상 시계열로 바꾸는 기법, 예측 후 과거 값을 더해 되돌림
- 시계열이 자신의 지연본과 상관이 있으면 자기상관이 있다고 함.

In [ ]:
# 데이터 불러오기
import pandas as pd
from pathlib import Path

path = Path("datasets/ridership/CTA_-_Ridership_-_Daily_Boarding_Totals.csv")
df = pd.read_csv(path, parse_dates=["service_date"])
df.columns = ["date", "day_type", "bus", "rail", "total"]
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1)
df = df.drop_duplicates()

# 차분과 자기상관
diff_7 = df[["bus", "rail"]].diff(7)["2019-03":"2019-05"]

# 오차 지표 MAE(평균 절대 오차)
print(diff_7.abs().mean())

# MAPE(평균 절대 백분율 오차)
targets = df[["bus", "rail"]]["2019-03":"2019-05"]
print((diff_7/targets).abs().mean())

# 연간 계절성과 장기 추세
period = slice("2001", "2019")
df_monthly = df.select_dtypes(include="number").resample('ME').mean()
rolling_average_12_months = df_monthly.loc[period].rolling(window=12).mean()

# 12개월 차분
df_monthly.diff(12)[period].plot(grid=True, marker=".", figsize=(8, 3))

#### ARMA 모델 계열
- 과거 값의 가중합으로 예측하고, 최근 예측 오차들의 가중합(이동평균)을 더해 보정
- 식: ŷ(t) = Σᵢ₌₁ᵖ αᵢ·y(t−i) + Σᵢ₌₁ᵠ θᵢ·ε(t−i), 단 ε(t) = y(t) − ŷ(t)
- ARIMA: 시계열을 더 정상으로 만들려고 d번 차분한 뒤 일반 ARMA 적용한다. 예측 시, ARMA로 예측하고, 차분으로 뺐던 항을 다시 더해 원래 스케일로 복원
- SARIMA(계절성 ARIMA): ARIMA와 같은 방식으로 모델링하되, 주어진 주기의 계절 성분도 똑같은 ARIMA 방식으로 모델링한다.
- 한 날의 결과로 모델을 판단하지 말고 여러 날의 평균 성능으로 평가해야 함.

In [1]:
%pip install statsmodels.tsa.arima.model

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement statsmodels.tsa.arima.model (from versions: none)

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for statsmodels.tsa.arima.model


In [ ]:
from statsmodels.tsa.arima.model import ARIMA

origin, today = "2019-01-01", "2019-05-31"
rail_series = df.loc[origin:today]["rail"].asfreq("D")
model = ARIMA(rail_series,
              order=(1, 0, 0),
              seasonal_order=(0, 1, 1, 7))
model = model.fit()
y_pred = model.forecast()  # 427,758.6

In [ ]:
# 3~5월 매일 예측
origin, start_date, end_date = "2019-01-01", "2019-03-01", "2019-05-31"
time_period = pd.date_range(start_date, end_date)
rail_series = df.loc[origin:end_date]["rail"].asfreq("D")
y_preds = []
for today in time_period.shift(-1):
    model = ARIMA(rail_series[origin:today],  # "오늘"까지의 데이터로 훈련
                  order=(1, 0, 0),
                  seasonal_order=(0, 1, 1, 7))
    model = model.fit()  # 매일 재훈련
    y_pred = model.forecast().iloc[0]
    y_preds.append(y_pred)

y_preds = pd.Series(y_preds, index=time_period)
mae = (y_preds - rail_series[time_period]).abs().mean()  # 32,040.7

#### 머신러닝 모델용 데이터 준비

In [ ]:
# 시계열을 가능한 모든 창과 타깃 쌍으로 자르는 커스텀 데이터셋
import torch
class TimeSeriesDataset(torch.utils.data.Dataset):
    def __init__(self, series, window_length):
        self.series = series
        self.window_length = window_length

    def __len__(self):
        return len(self.series) - self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        end = idx + self.window_length  # 창 다음 첫 인덱스
        window = self.series[idx : end]
        target = self.series[end]
        return window, target

In [ ]:
my_series = torch.tensor([[0], [1], [2], [3], [4], [5]])
my_dataset = TimeSeriesDataset(my_series, window_length=3)
for window, target in my_dataset:
	print("Window:", window, "Target:", target)

In [ ]:
# DataLoader로 배치 만들기
from torch.utils.data import DataLoader
torch.manual_seed(0)
my_loader = DataLoader(my_dataset, batch_size=2, shuffle=True)
for X, y in my_loader:
	print("X:", X, "y:", y)
rail_train = torch.FloatTensor(df[["rail"]]["2016-01":"2018-12"].values / 1e6)
rail_valid = torch.FloatTensor(df[["rail"]]["2019-01":"2019-05"].values / 1e6)
rail_test = torch.FloatTensor(df[["rail"]]["2019-06":].values / 1e6)

In [ ]:
# 데이터셋과 데이터 로더
window_length = 56
train_set = TimeSeriesDataset(rail_train, window_length)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
valid_set = TimeSeriesDataset(rail_valid, window_length)
valid_loader = DataLoader(valid_set, batch_size=32)
test_set = TimeSeriesDataset(rail_test, window_length)
test_loader = DataLoader(test_set, batch_size=32)

#### 선형 모델로 예측

In [ ]:
import torch.nn as nn
import torchmetrics

torch.manual_seed(42)
model = nn.Sequential(nn.Flatten(), nn.Linear(window_length, 1)).to(device)
loss_fn = nn.HuberLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.003, momentum=0.9)
metric = torchmetrics.MeanAbsoluteError().to(device)
[...] # 10장의 train() 함수 등으로 훈련

#### 간단한 RNN으로 예측
- 메모리 셀을 타임 스텝마다 적용해 은닉 상태를 갱신하고, 마지막 은닉 상태를 Linear에 넣어 예측 (시퀀스 투 벡터)

In [ ]:
class SimpleRnnModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.memory_cell = nn.Sequential(
            nn.Linear(input_size + hidden_size, hidden_size),
            nn.Tanh()
        )
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        batch_size, window_length, dimensionality = X.shape
        X_time_first = X.transpose(0, 1)
        H = torch.zeros(batch_size, self.hidden_size, device=X.device)
        for X_t in X_time_first:
            XH = torch.cat((X_t, H), dim=1)
            H = self.memory_cell(XH)
        return self.output(H)

torch.manual_seed(42)
univar_model = SimpleRnnModel(input_size=1, hidden_size=32, output_size=1)

In [ ]:
# nn.RNN 모듈로 단순화
class SimpleRnnModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, last_state = self.rnn(X)
        return self.output(outputs[:, -1])

#### 깊은 RNN으로 예측
- 셀을 여러 층으로 쌓아 깊은 RNN이 된다.

In [ ]:
self.rnn = nn.RNN(input_size, hidden_size, num_layers=3, batch_first=True)

#### 다변량 시계열 예측

In [ ]:
df_mulvar = df[["rail", "bus"]] / 1e6  # 철도와 버스를 모두 입력으로
df_mulvar["next_day_type"] = df["day_type"].shift(-1)  # 내일의 유형을 알고 있음
df_mulvar = pd.get_dummies(df_mulvar, dtype=float)  # 요일 유형 원-핫 인코딩

In [ ]:
# 분할과 텐서 변환
mulvar_train = torch.FloatTensor(df_mulvar["2016-01":"2018-12"].values / 1e6)
mulvar_valid = torch.FloatTensor(df_mulvar["2019-01":"2019-05"].values / 1e6)
mulvar_test = torch.FloatTensor(df_mulvar["2019-06":].values / 1e6)

In [ ]:
# 데이터셋
class MulvarTimeSeriesDataset(TimeSeriesDataset):
    def __getitem__(self, idx):
        window, target = super().__getitem__(idx)
        return window, target[:1]

In [ ]:
mulvar_train_set = MulvarTimeSeriesDataset(mulvar_train, window_length)
mulvar_train_loader = DataLoader(mulvar_train_set, batch_size=32, shuffle=True)
[...]  # 검증, 테스트 세트도 같은 방식으로

In [ ]:
# 모델
torch.manual_seed(42)
mulvar_model = SimpleRnnModel(input_size=5, hidden_size=32, output_size=1)
mulvar_model = mulvar_model.to(device)

#### 여러 타임 스텝 앞 예측
- 지금까지는 다음 스텝 하나만 예측했지만, 타깃을 바꾸면 더 먼 미래 하나를 예측할 수 있다.
- 방법은 한 스텝씩 반복 예측하거나, 값을 한 번에 예측

In [ ]:
# 방법 1: 한 스텝씩 반복 예측
n_steps = 14
univar_model.eval()
with torch.no_grad():
    X = rail_valid[:window_length].unsqueeze(dim=0).to(device)
    for step_ahead in range(n_steps):
        y_pred_one = univar_model(X)
        X = torch.cat([X, y_pred_one.unsqueeze(dim=1)], dim=1)

    Y_pred = X[0, -n_steps:, 0]

In [ ]:
# 방법 2: 14개 값을 한 번에 예측
class ForecastAheadDataset(TimeSeriesDataset):
    def __len__(self):
        return len(self.series) - self.window_length - 14 + 1

    def __getitem__(self, idx):
        end = idx + self.window_length  # 창 다음 첫 인덱스
        window = self.series[idx : end]
        target = self.series[end : end + 14, 0]  # 0 = 철도 승객 수
        return window, target

In [ ]:
# 데이터셋, 로더, 모델
ahead_train_set = ForecastAheadDataset(mulvar_train, window_length)
ahead_train_loader = DataLoader(ahead_train_set, batch_size=32, shuffle=True)
[...]  # 검증, 테스트 세트도 같은 방식으로

torch.manual_seed(42)
ahead_model = SimpleRnnModel(input_size=5, hidden_size=32, output_size=14)
ahead_model = ahead_model.to(device)

In [ ]:
# 예측
ahead_model.eval()
with torch.no_grad():
    window = mulvar_valid[:window_length]  # 모양 [56, 5]
    X = window.unsqueeze(dim=0)            # 모양 [1, 56, 5]
    Y_pred = ahead_model(X.to(device))     # 모양 [1, 14]

#### 시퀀스 투 시퀀스 모델로 예측
- 매 타임 스텝마다 다음 output_size개 값을 예측하도록 훈련

In [ ]:
class Seq2SeqDataset(ForecastAheadDataset):
    def __getitem__(self, idx):
        end = idx + self.window_length  # 창 다음 첫 인덱스
        window = self.series[idx : end]
        target_period = self.series[idx + 1 : end + 14, 0]
        target = target_period.unfold(dimension=0, size=14, step=1)
        return window, target
torch.tensor([0, 1, 2, 3, 4, 5]).unfold(dimension=0, size=4, step=1)

# 데이터셋과 로더
seq_train_set = Seq2SeqDataset(mulvar_train, window_length)
seq_train_loader = DataLoader(seq_train_set, batch_size=32, shuffle=True)
[...]  # 검증, 테스트 세트도 같은 방식으로

In [ ]:
# 모델
class Seq2SeqRnnModel(SimpleRnnModel):
    def forward(self, X):
        outputs, last_state = self.rnn(X)
        return self.output(outputs)

torch.manual_seed(42)
seq_model = Seq2SeqRnnModel(input_size=5, hidden_size=32, output_size=14)
seq_model = seq_model.to(device)

In [ ]:
# 훈련과 예측
seq_model.eval()
with torch.no_grad():
    some_window = mulvar_valid[:window_length]  # 모양 [56, 5]
    X = some_window.unsqueeze(dim=0)            # 모양 [1, 56, 5]
    Y_preds = seq_model(X.to(device))           # 모양 [1, 56, 14]
    Y_pred = Y_preds[:, -1]                     # 모양 [1, 14]

#### 긴 시퀀스 다루기
- 긴 시퀀스 = 펼친 RNN이 매우 깊어짐 -> 불안정한 그래디언트와 초반 입력 망각이라는 두 문제가 생김

#### 불안정한 그래디언트 문제 대처
| 기법 | RNN에서의 결과 |
| --- | --- |
| ReLU 등 비포화 함수 | 같은 가중치가 반복되어 출력이 폭주하기 쉬움 → tanh(포화)나 작은 학습률 |
| 그레이디언트 폭주 | 그레이디언트 클리핑 |
| 배치 정규화 | 스텝 사이에는 비효과적, 층 사이에서만 약간 도움 |
| 층 정규화 | 활성화 앞에 매 스텝 적용하면 BN보다 낫지만, nn.RNN이 지원하지 않아 수동 구현 필요. 게이트 RNN에서 더 잘 맞음 |
| 드롭아웃 | nn.RNN은 층 사이만 지원(dropout=). 스텝 사이는 커스텀 구현 |
| 오차 막대 | MC드롭아웃 |

In [ ]:
# 앞서 만든 첫 번째 SimpleRnnModel의 메모리 셀에 nn.LayerNorm을 추가
self.memory_cell = nn.Sequential(
    nn.Linear(input_size + hidden_size, hidden_size),
    nn.LayerNorm(hidden_size),
    nn.Tanh()
)

#### 단기 기억 문제 해결
- 시간이 지나면 RNN의 상태에는 맨 처음 입력의 흔적이 거의 남지 않고, 이는 치명적인 문제가 될 수 있다.
- 장기 기억을 가진 여러 종류의 셀이 도입되었다.

#### LSTM 셀
- LSTM = 상태를 단기 h(t)와 장기 c(t)로 나누고, 세 개의 게이트(삭제 f, 입력 i, 츨력 o)로 장기 상태에 무엇을 버리고, 더하고, 읽을지 학습하는 셀
- 핵심 식 c(t) = f(t)⊗c(t−1) + i(t)⊗g(t), h(t) = y(t) = o(t)⊗tanh(c(t)), 게이트는 시그모이드(0~1), 후보 기억 g(t)는 tanh
- nn.RNN을 nn.LSTM으로 바꾸면 되고, 보통 수렴이 빠르며 더 긴 패턴을 잡음
- 스텝마다 LN이나 드롭아웃을 넣으려면 nn.LSTMCell로 루프를 직접 구현하고, 상태를 (H, C)쌍으로 다룸.

In [ ]:
class LstmModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.memory_cell = nn.LSTMCell(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        batch_size, window_length, dimensionality = X.shape
        X_time_first = X.transpose(0, 1)
        H = torch.zeros(batch_size, self.hidden_size, device=X.device)
        C = torch.zeros(batch_size, self.hidden_size, device=X.device)
        for X_t in X_time_first:
            H, C = self.memory_cell(X_t, (H, C))
        return self.output(H)

#### GRU 셀
- GRU = LSTM을 단순화한 셀로, 상태 하나(h), 게이트 제어기 z(t)로 삭제와 입력을 함께 제어, 출력 게이트 없음, 대신 리셋 게이트 r(t)가 이전 상태 중 주 층에 보여 줄 부분을 고름.
- 핵심 식: h(t) = z(t)⊗h(t−1) + (1−z(t))⊗g(t)
- 구현은 nn.RNN/nn.LSTM을 nn.GRU로 교체하고, 커스텀이면 nn.GRUCell을 쓴다.

#### 1D 합성곱층으로 시퀀스 처리
- 1D 합성곱층 = 커널이 시퀀스 위를 슬라이딩하며 짧은 순차 패턴을 감지. nn.Conv1d는 [배치, 특성, 길이]를 기대하므로 마지막 두 차원을 바꿔야 함.
- 스트라이드 > 1 이나 valid ㅠㅐ딩이면 출력이 짧아지므로 타깃도 맞춰야 함.

In [ ]:
# 모델: Conv1d + GRU + Linear
class DownsamplingModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.conv = nn.Conv1d(input_size, hidden_size, kernel_size=4, stride=2)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        Z = X.permute(0, 2, 1)  # 시간을 공간 차원처럼 취급
        Z = self.conv(Z)
        Z = Z.permute(0, 2, 1)  # 시간과 특성 차원을 다시 바꿈
        Z = torch.relu(Z)
        Z, _states = self.gru(Z)
        return self.linear(Z)

torch.manual_seed(42)
dseq_model = DownsamplingModel(input_size=5, hidden_size=32, output_size=14)
dseq_model = dseq_model.to(device)
# 타깃 조정
class DownsampledDataset(Seq2SeqDataset):
    def __getitem__(self, idx):
        window, target = super().__getitem__(idx)
        return window, target[3::2]  # 처음 3개 자르고 다운샘플링

window_length = 112
dseq_train_set = DownsampledDataset(rail_train, window_length)
dseq_train_loader = DataLoader(dseq_train_set, batch_size=32, shuffle=True)
[...]  # 검증, 테스트 세트도 같은 방식으로

#### WaveNet
- WaveNet = 팽창(dilated) 1D 합성곱층을 쌓되 팽창률을 1,2,4,8,...로 2배씩 늘려 적은 층으로 매우 큰 수용장을 얻는 구조
- 인과적 패딩으로 길이를 유지하고 미래를 엿보지 않게 함.
- 길이가 유지되므로 타깃을 자르거나 다운샘플링할 필요 없이 Seq2SeqRnnModel 용 데이터 로더를 그대로 씀.

In [ ]:
# 인과적 합성곱
import torch.nn.functional as F

class CausalConv1d(nn.Conv1d):
    def forward(self, X):
        padding = (self.kernel_size[0] - 1) * self.dilation[0]
        X = F.pad(X, (padding, 0))
        return super().forward(X)

In [ ]:
# WaveNet 모델
class WavenetModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        layers = []
        for dilation in (1, 2, 4, 8) * 2:
            conv = CausalConv1d(input_size, hidden_size, kernel_size=2,
                                dilation=dilation)
            layers += [conv, nn.ReLU()]
            input_size = hidden_size
        self.convs = nn.Sequential(*layers)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        Z = X.permute(0, 2, 1)
        Z = self.convs(Z)
        Z = Z.permute(0, 2, 1)
        return self.output(Z)

torch.manual_seed(42)
wavenet_model = WavenetModel(input_size=5, hidden_size=32, output_size=14)
wavenet_model = wavenet_model.to(device)

#### Exercises
1. 시퀀스 투 시퀀스 / 시퀀스 투 벡터 / 벡터 투 시퀀스 RNN의 응용
- 시퀀스 투 시퀀스: 시계열 예측(매 스텝 다음 값), 음성 인식, 품사 태깅, 영상의 프레임별 레이블링, 음악 생성, 텍스트 생성
- 시퀀스 투 벡터: 감성 분석(리뷰 → 점수), 음악 장르 분류, 센서/생체 신호(예: 뇌파) 분류, 이용 기록으로 선호도 예측, 시계열의 다음 값 하나 예측
- 벡터 투 시퀀스: 이미지 캡션 생성, 어떤 파라미터(장르, 스타일 등)로부터 음악이나 멜로디 생성

2. RNN 층의 입력과 출력 차원
- 입력은 3차원
- 첫째: 배치크기, 둘째: 타임 스텝, 셋째: 스텝당 입력 특성 수
- 출력도 3차원
- outputs: [배치, 타임 스텝, 은닉 크기], last_state: [층 수, 배치, 은닉 크기]

4. 일변량 일별 시계열로 다음 7일 예측
- 시퀀스 투 시퀀스 RNN을 사용
- 출력 크기를 7로 하고, 타깃은 각 스텝에서의 다음 7개 값으로 만든다.

5. RNN 훈련의 주요 어려움과 대처
- 불안정한 그레이디언트(소실/폭주): 좋은 초기화, 더 작은 학습률, tanh 같은 포화 활성화, 그레이디언트 클리핑, 스텝별 층 정규화, 순환 드롭아웃(BN은 스텝 사이에서는 비효과적)
- 제한된 단기 기억: LSTM, GRU 셀로 교체, 입력 시퀀스를 1D 합성곱(스트라이드) 으로 줄이기, WaveNet류 팽창 합성곱
- 긴 시퀀스에서 느린 훈련: 위와 같은 단축, 시퀀스 투 시퀀스 훈련으로 그레이디언트 경로 단축

7. RNN에서 1D 합성곱층을 쓰는 이유
- 시퀀스를 짧게 줄여 주기 때문이다. 줄어든 시퀀스에서는 RNN이 더 긴 패턴을 감지할 수 있어 단기 기억 한계를 덜 겪는다.

8. 비디오 분류에서 쓸 수 있는 신경망
- 프레임별 CNN + RNN
- 3D 합성곱 신경망
- 프레임 CNN + 1D 합성곱
- 트랜스포머

In [ ]:
# Exercises 3. 깊은 시퀀스 투 시퀀스 RNN 만들기
class DeepSeq2Seq(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=3):
        super().__init__()
        self.rnn = nn.GRU(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, _ = self.rnn(X)
        return self.output(outputs)   # [배치, 시간, output_size]

In [ ]:
# Exercises 9.
# 데이터셋 수정
import torch
from torch.utils.data import Dataset, DataLoader

class MultiTargetSeq2SeqDataset(Dataset):
    def __init__(self, series_df, window_length=56, ahead=14):
        # 입력- bus, rail 및 요일/휴일 원-핫 인코딩 등 모든 피처 포함
        self.X = torch.tensor(series_df.values, dtype=torch.float32)
        # 타깃- 예측 대상인 rail과 bus 두 컬럼만 추출
        self.targets = torch.tensor(series_df[["rail", "bus"]].values, dtype=torch.float32)
        self.window_length = window_length
        self.ahead = ahead

    def __len__(self):
        return len(self.X) - self.window_length - self.ahead + 1

    def __getitem__(self, idx):
        # 입력 윈도우: [window_length, n_features]
        window = self.X[idx : idx + self.window_length]
        
        # 각 타임스텝(t)마다 미래 14일(t+1 ~ t+14)의 [rail, bus] 값을 묶음
        target_steps = [
            self.targets[idx + t + 1 : idx + t + 1 + self.ahead]
            for t in range(self.window_length)
        ]
        target = torch.stack(target_steps)
        
        # 모델 출력과 맞추기 위해 마지막 두 차원(14, 2)을 평탄화(flatten)
        target = target.reshape(self.window_length, -1)
        return window, target

In [ ]:
# 모델 수정
import torch.nn as nn

class SimpleRnnModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size, hidden_size, num_layers=num_layers, batch_first=True
        )
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, _ = self.rnn(X)
        return self.output(outputs[:, -1])

class Seq2SeqRnnModel(SimpleRnnModel):
    def forward(self, X):
        outputs, _ = self.rnn(X)
        # outputs: [batch_size, window_length, hidden_size]
        # 반환값: [batch_size, window_length, 28]
        return self.output(outputs)

In [ ]:
# 디바이스 코드
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

In [ ]:
# 모델 초기화, 학습 및 평가
# 데이터셋 및 로더 생성
window_length = 56
train_set = MultiTargetSeq2SeqDataset(mulvar_train, window_length=window_length, ahead=14)
valid_set = MultiTargetSeq2SeqDataset(mulvar_valid, window_length=window_length, ahead=14)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=32)

# 모델 생성
n_inputs = mulvar_train.shape[1]
model = Seq2SeqRnnModel(
    input_size=n_inputs,
    hidden_size=32,
    output_size=28,  # 14일 * 2개 변수(rail, bus)
    num_layers=1
).to(device)

# 학습 루프
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
loss_fn = nn.HuberLoss()

for epoch in range(20):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_pred = model(X_batch)          # [batch_size, 56, 28]
        loss = loss_fn(y_pred, y_batch)  # 전체 타임스텝에 대해 손실 계산
        loss.backward()
        optimizer.step()

# 검증 세트 평가
model.eval()
total_abs_err = torch.zeros(2, device=device)  # [rail 오차 합, bus 오차 합]
total_samples = 0

with torch.no_grad():
    for X_batch, y_batch in valid_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)
        
        # 마지막 타임스텝(:, -1)만 추출 후 [batch_size, 14, 2]로 복원
        last_pred = y_pred[:, -1].reshape(-1, 14, 2)
        last_true = y_batch[:, -1].reshape(-1, 14, 2)
        
        # 스케일링(1e6) 복원 후 rail(인덱스 0), bus(인덱스 1) 각각의 절대 오차 합산
        abs_err = torch.abs(last_pred - last_true) * 1e6
        total_abs_err += abs_err.sum(dim=(0, 1))
        total_samples += X_batch.size(0) * 14

mae_rail, mae_bus = (total_abs_err / total_samples).tolist()
print(f"Validation MAE - Rail: {mae_rail:,.0f}, Bus: {mae_bus:,.0f}")

In [ ]:
# Exercises 10.
import urllib.request
import tarfile
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# JSB Chorales 데이터셋 다운로드 및 압축 해제
url = "https://github.com/ageron/data/raw/main/jsb_chorales.tgz"
tgz_path = Path("jsb_chorales.tgz")
data_dir = Path("jsb_chorales")

if not data_dir.exists():
    urllib.request.urlretrieve(url, tgz_path)
    with tarfile.open(tgz_path, "r:gz") as tar:
        tar.extractall()

def load_chorales(split):
    files = sorted((data_dir / split).glob("chorale_*.csv"))
    return [pd.read_csv(f).values for f in files]

train_chorales = load_chorales("train")
valid_chorales = load_chorales("valid")
test_chorales = load_chorales("test")

# 고유 음표(MIDI 인덱스) 확인 및 0~46 연속 인덱스로 매핑 테이블 생성
all_notes = set()
for split_data in (train_chorales, valid_chorales, test_chorales):
    for chorale in split_data:
        all_notes.update(chorale.flatten())

# 0(무음)은 0번 인덱스에 고정, 나머지 46개 음표는 1~46번에 매핑
sorted_notes = sorted(all_notes)
note_to_idx = {note: idx for idx, note in enumerate(sorted_notes)}
idx_to_note = {idx: note for idx, note in enumerate(sorted_notes)}
vocab_size = len(sorted_notes)  # 47

In [ ]:
class BachChoralesDataset(Dataset):
    def __init__(self, chorales, note_to_idx, window_size=128, stride=16):
        self.windows = []
        for chorale in chorales:
            # MIDI 번호를 0~46 인덱스로 변환 후 1차원으로 펼침
            mapped = np.vectorize(note_to_idx.get)(chorale).flatten()
            # 슬라이딩 윈도우로 학습 샘플 추출
            for start in range(0, len(mapped) - window_size, stride):
                seq = mapped[start : start + window_size + 1]
                self.windows.append(torch.tensor(seq, dtype=torch.long))

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        seq = self.windows[idx]
        return seq[:-1], seq[1:] 

window_size = 128
train_set = BachChoralesDataset(train_chorales, note_to_idx, window_size=window_size, stride=16)
valid_set = BachChoralesDataset(valid_chorales, note_to_idx, window_size=window_size, stride=32)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=64)

In [ ]:
# Conv1D + LSTM
class CausalConv1d(nn.Module):
    # 미래 시점 누출을 막기 위해 왼쪽에만 패딩을 주는 1D 합성곱
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation)

    def forward(self, x):
        # x: [batch, channels, seq_len]
        x = F.pad(x, (self.padding, 0))
        return self.conv(x)

class BachGeneratorModel(nn.Module):
    def __init__(self, vocab_size=47, embed_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # WaveNet 스타일의 팽창(dilated) 인과적 합성곱 블록
        self.conv_blocks = nn.Sequential(
            CausalConv1d(embed_dim, 32, kernel_size=2, dilation=1),
            nn.ReLU(),
            CausalConv1d(32, 32, kernel_size=2, dilation=2),
            nn.ReLU(),
            CausalConv1d(32, 32, kernel_size=2, dilation=4),
            nn.ReLU(),
            CausalConv1d(32, 32, kernel_size=2, dilation=8),
            nn.ReLU(),
        )
        
        self.lstm = nn.LSTM(input_size=32, hidden_size=128, num_layers=1, batch_first=True)
        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):
        # x: [batch, seq_len]
        emb = self.embedding(x)
        conv_in = emb.transpose(1, 2)
        conv_out = self.conv_blocks(conv_in)
        lstm_in = conv_out.transpose(1, 2)
        lstm_out, _ = self.lstm(lstm_in)
        logits = self.fc(lstm_out)
        return logits

In [ ]:
# 모델 학습
model = BachGeneratorModel(vocab_size=vocab_size, embed_dim=16).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(25):
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)  # [batch, seq_len, vocab_size]
        loss = criterion(logits.reshape(-1, vocab_size), y_batch.reshape(-1))
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)

    model.eval()
    valid_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits.reshape(-1, vocab_size), y_batch.reshape(-1))
            valid_loss += loss.item() * X_batch.size(0)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss/len(train_set):.4f} | Valid Loss: {valid_loss/len(valid_set):.4f}")

In [ ]:
def generate_chorale(model, seed_chorale, num_new_steps=64, temperature=0.8):
    model.eval()
    # 시드 코랄의 앞부분을 인덱스 시퀀스로 변환
    seed_mapped = np.vectorize(note_to_idx.get)(seed_chorale).flatten()
    generated = torch.tensor(seed_mapped, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        for _ in range(num_new_steps * 4):
            logits = model(generated)
            # 마지막 시점의 로짓에 temperature 적용 후 확률 분포에서 샘플링
            next_logits = logits[:, -1, :] / temperature
            probs = F.softmax(next_logits, dim=-1)
            next_note = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_note], dim=1)

    # 인덱스를 원래 MIDI 피아노 번호로 복원하고 [타임스텝, 4] 형태로 재구성
    gen_indices = generated.squeeze(0).cpu().numpy()
    gen_notes = np.vectorize(idx_to_note.get)(gen_indices)
    return gen_notes.reshape(-1, 4)

# 테스트 코랄의 처음 8스텝을 시드로 사용해 56스텝(총 64스텝) 생성
seed = test_chorales[0][:8]
new_chorale = generate_chorale(model, seed_chorale=seed, num_new_steps=56, temperature=0.8)
print("Generated chorale shape:", new_chorale.shape)
print("First 12 time steps:\n", new_chorale[:12])

In [ ]:
# Exercises 11.
# 데이터 다운로드 및 시퀀스 전처리
import json
import urllib.request
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# 클래스별 NDJSON 파일 다운로드
classes = ["ant", "axe", "bat", "bicycle", "cat"]
data_dir = Path("quickdraw_data")
data_dir.mkdir(exist_ok=True)

base_url = "https://storage.googleapis.com/quickdraw_dataset/full/simplified"
max_samples_per_class = 10_000

sketches = []
labels = []

for class_idx, class_name in enumerate(classes):
    file_path = data_dir / f"{class_name}.ndjson"
    if not file_path.exists():
        urllib.request.urlretrieve(f"{base_url}/{class_name}.ndjson", file_path)
    
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= max_samples_per_class:
                break
            item = json.loads(line)
            sketches.append(item["drawing"])
            labels.append(class_idx)

# 스케치 -> [총 시퀀스 길이, 3] 텐서 변환 함수
def sketch_to_tensor(drawing):
    stroke_tensors = []
    for stroke in drawing:
        x = np.array(stroke[0], dtype=np.float32) / 255.0
        y = np.array(stroke[1], dtype=np.float32) / 255.0
        # 현재 획에서의 진행률
        progress = np.linspace(0.0, 1.0, len(x), dtype=np.float32)
        
        # [획의 점 개수, 3] 형태로 결합
        stroke_features = np.stack([x, y, progress], axis=-1)
        stroke_tensors.append(stroke_features)
    
    # 모든 획을 시간 순서대로 하나의 시퀀스로 연결: [전체 점 개수, 3]
    full_seq = np.concatenate(stroke_tensors, axis=0)
    return torch.tensor(full_seq, dtype=torch.float32)

X_tensors = [sketch_to_tensor(s) for s in sketches]
y_tensors = torch.tensor(labels, dtype=torch.long)

In [ ]:
class QuickDrawDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)
    # 가장 긴 스케치 길이에 맞춰 0으로 패딩: [batch_size, max_len, 3]
    padded_seqs = pad_sequence(sequences, batch_first=True, padding_value=0.0)
    labels = torch.stack(labels)
    return padded_seqs, lengths, labels

full_dataset = QuickDrawDataset(X_tensors, y_tensors)
train_size = int(0.8 * len(full_dataset))
valid_size = len(full_dataset) - train_size
train_set, valid_set = random_split(full_dataset, [train_size, valid_size])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_set, batch_size=64, shuffle=False, collate_fn=collate_fn)

In [ ]:
# Conv1D + Packed LSTM
class QuickDrawClassifier(nn.Module):
    def __init__(self, num_classes, input_size=3, hidden_size=128):
        super().__init__()
        # 시퀀스 길이를 유지하는 1D 합성곱 블록
        self.conv = nn.Sequential(
            nn.Conv1d(input_size, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )
        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        # x: [batch, max_len, 3] -> Conv1d 입력을 위해 [batch, 3, max_len]으로 전치
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.transpose(1, 2)  # [batch, max_len, 64]

        # 패딩된 부분을 건너뛰도록 시퀀스 패킹
        packed = pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        
        # 마지막 LSTM 레이어의 최종 은닉 상태 h_n[-1]: [batch, hidden_size]
        logits = self.fc(h_n[-1])
        return logits

In [ ]:
# 모델 학습 및 검증
model = QuickDrawClassifier(num_classes=len(classes)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    train_loss, train_correct, total_train = 0.0, 0, 0
    for X_batch, lengths, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        logits = model(X_batch, lengths)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * y_batch.size(0)
        train_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total_train += y_batch.size(0)

    model.eval()
    valid_correct, total_valid = 0, 0
    with torch.no_grad():
        for X_batch, lengths, y_batch in valid_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch, lengths)
            valid_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total_valid += y_batch.size(0)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss/total_train:.4f} | "
        f"Train Acc: {train_correct/total_train:.2%} | "
        f"Valid Acc: {valid_correct/total_valid:.2%}"
    )